<div align="center">
<h1>🔐 KRİPTOQRAFİYA KURSU</h1>
<h2>Məşğələ 30 — Müasir məxfilik texnologiyaları</h2>
<p><b>ZKP · homomorf şifrələmə · blokçeyn · QKD</b></p>
<p>Yekun praktika · 2 saat · yoxlanmış versiya (2026-07-21)</p>
</div>


## 📑 Mündəricat

1. Mühit və icra müqaviləsi
2. Sıfır-bilik sübutları: Schnorr sigma-protokolu
3. Qismən və tam homomorf şifrələmə
4. Heş-zəncir və Merkle daxilolma sübutu
5. BB84-də intercept-resend və QBER
6. MPC, DID və CBDC üçün təhlükəsizlik sərhədləri
7. Yekun tapşırıq və özünü-yoxlama


## 🎯 Məşğələnin məqsədləri

Məşğələnin sonunda tələbə:

- ZKP üçün tamlıq, soundness və zero-knowledge anlayışlarını konkret modelə görə ayırır;
- Schnorr sigma-protokolunun düzgün və səhv transkriptlərini yoxlayır;
- Paillier-in additiv homomorfizmini və malleability məhdudiyyətini nümayiş etdirir;
- heş-zəncirin yalnız tamper-evidence verdiyini, konsensus vermədiyini izah edir;
- BB84 intercept-resend modelində QBER-i ölçür və post-processing sərhədini göstərir;
- texnologiya, protokol, sistem siyasəti və gələcək proqnozu bir-birinə qarışdırmır.


## 📚 İcra mühiti

Notebook yalnız Python standart kitabxanasından istifadə edir. Runtime `pip install`, şəbəkə,
`input()` və işçi qovluğun dəyişdirilməsi yoxdur. Nümunələr tədris üçündür: kiçik parametrlər,
deterministik test RNG-si və sadələşdirilmiş attacker modelləri production təhlükəsizliyi vermir.


In [ ]:
# İcra müqaviləsi: xarici paket və runtime quraşdırma yoxdur.
import platform
import sys

assert sys.version_info >= (3, 11), "Python 3.11+ tələb olunur"
ENVIRONMENT = {
    "python": platform.python_version(),
    "implementation": platform.python_implementation(),
    "external_dependencies": 0,
    "network_required": False,
}
print("İcra mühiti:", ENVIRONMENT)


## 🔧 Hazırlıq

Hər kod bölməsi əvvəlcə müsbət nəticəni, sonra ən azı bir mənfi halı yoxlayır. `assert`
uğursuz olarsa notebook dayanmalıdır; xəta `skip` və ya geniş `except` ilə gizlədilmir.


### 1.1 İdxallar və köməkçi funksiyalar

Kriptoqrafik protokollar üçün production təsadüfiliyi `secrets` və ya audit olunmuş kitabxana
tələb edir. Buradakı `random.Random` yalnız nəticəni təkrar istehsal etmək və statistik toy-modeli
yoxlamaq üçündür.


In [ ]:
from dataclasses import dataclass
from hashlib import sha256
from math import gcd, lcm
import json
import random


def canonical_bytes(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8")


def digest_hex(data):
    if isinstance(data, str):
        data = data.encode("utf-8")
    return sha256(data).hexdigest()


TEST_SEED = 32026
print("Deterministik test seed-i:", TEST_SEED)


### 1.2 Təkrar istehsal və sərhədlər

Kod fayl yaratmır və cari qovluğu dəyişmir. Hər model öz nəticəsini yaddaşda saxlayır. Eyni
seed eyni tədris nəticəsini verir; bu xüsusiyyət gizli açar yaratmaq üçün uyğun deyil.


In [ ]:
RUN_CONTRACT = {
    "filesystem_writes": 0,
    "working_directory_changes": 0,
    "interactive_prompts": 0,
    "runtime_installs": 0,
    "toy_models_are_production": False,
}
assert all(RUN_CONTRACT[key] == 0 for key in (
    "filesystem_writes", "working_directory_changes", "interactive_prompts", "runtime_installs"
))
assert RUN_CONTRACT["toy_models_are_production"] is False
print("İcra müqaviləsi təsdiqləndi:", RUN_CONTRACT)


## 📖 Kursla əlaqə

Bu praktika əvvəlki mövzuları yeni sistemlərdə birləşdirir: modular hesab və diskret loqarifm
Schnorr üçün; açıq açarlı şifrələmə Paillier üçün; heş funksiyaları Merkle ağacı üçün;
ehtimal və autentifikasiya isə BB84 təhlükəsizlik müzakirəsi üçün lazımdır.


| Bu praktikanın bölməsi | Əvvəlki bilik | Burada yoxlanan sərhəd |
|---|---|---|
| Schnorr sigma-protokolu | modul qüvvət, DLP | transkript ≠ production ZK sistemi |
| Paillier | Euler/CRT, açıq açar | homomorfizm ≠ autentifikasiya |
| Merkle/heş-zəncir | SHA-2, rəqəmsal imza | tamper-evidence ≠ konsensus |
| BB84 | ehtimal, autentifikasiya | sifting ≠ yekun məxfi açar |


## 🔐 Sıfır-bilik sübutları (ZKP)


### 2.1 Dəqiq anlayış və tarix

İnteraktiv sübut sistemi konkret dil, təhlükəsizlik parametri və verifier modeli üçün müəyyən edilir:

- **Tamlıq (completeness):** doğru ifadə və düzgün witness olduqda dürüst verifier qəbul edir.
- **Soundness:** yanlış ifadə üçün fırıldaqçı prover-in qəbul ehtimalı parametr artdıqca cüzi olur.
- **Zero-knowledge:** verifier-in gördüyü məlumat uyğun simulator tərəfindən witness olmadan yaradıla bilir.

Goldwasser, Micali və Rackoff anlayışı 1985-ci il STOC məqaləsində təqdim ediblər. 2012-ci il
Turing mükafatı Goldwasser və Micali-yə verilib; Rackoff həmin mükafatın laureatı deyil.


### 2.2 Schnorr sigma-protokolu — icra olunan toy-model

Prover `y = g^x mod p` açıq dəyərinə uyğun gizli `x` witness-ini bildiyini göstərir:

1. `r ← Z_q`, `t = g^r` commitment;
2. verifier təsadüfi `c ← Z_q` challenge göndərir;
3. prover `s = r + cx mod q` cavabını verir;
4. verifier `g^s = t·y^c mod p` bərabərliyini yoxlayır.

`p=23, q=11` parametrləri qəsdən kiçikdir və təhlükəsiz deyil. Bir transkript yalnız bu
riyazi invariantı nümayiş etdirir; autentifikasiya, Fiat–Shamir çevrilməsi və production
parametrləri ayrıca protokol tələb edir.


In [ ]:
P, Q, G = 23, 11, 2
assert pow(G, Q, P) == 1 and G != 1


@dataclass(frozen=True)
class SchnorrTranscript:
    commitment: int
    challenge: int
    response: int


def schnorr_public(secret):
    if not 1 <= secret < Q:
        raise ValueError("secret Z_q* daxilində olmalıdır")
    return pow(G, secret, P)


def schnorr_prove(secret, nonce, challenge):
    if not (0 <= nonce < Q and 0 <= challenge < Q):
        raise ValueError("nonce və challenge Z_q daxilində olmalıdır")
    return SchnorrTranscript(
        commitment=pow(G, nonce, P),
        challenge=challenge,
        response=(nonce + challenge * secret) % Q,
    )


def schnorr_verify(public, transcript):
    left = pow(G, transcript.response, P)
    right = transcript.commitment * pow(public, transcript.challenge, P) % P
    return left == right


SCHNORR_SECRET = 7
SCHNORR_PUBLIC = schnorr_public(SCHNORR_SECRET)
SCHNORR_TRANSCRIPT = schnorr_prove(SCHNORR_SECRET, nonce=4, challenge=6)
assert schnorr_verify(SCHNORR_PUBLIC, SCHNORR_TRANSCRIPT)

SCHNORR_TAMPERED = SchnorrTranscript(
    SCHNORR_TRANSCRIPT.commitment,
    SCHNORR_TRANSCRIPT.challenge,
    (SCHNORR_TRANSCRIPT.response + 1) % Q,
)
assert not schnorr_verify(SCHNORR_PUBLIC, SCHNORR_TAMPERED)
print("Schnorr: düzgün transkript qəbul, dəyişdirilmiş cavab rədd edildi")


In [ ]:
def simulate_honest_verifier_transcript(public, rng):
    # c və s seçilir, sonra t elə qurulur ki, yoxlama bərabərliyi ödənsin.
    challenge = rng.randrange(Q)
    response = rng.randrange(Q)
    commitment = pow(G, response, P) * pow(pow(public, challenge, P), -1, P) % P
    return SchnorrTranscript(commitment, challenge, response)


def extract_from_reused_commitment(first, second):
    if first.commitment != second.commitment or first.challenge == second.challenge:
        raise ValueError("eyni commitment və fərqli challenge tələb olunur")
    numerator = (first.response - second.response) % Q
    denominator = (first.challenge - second.challenge) % Q
    return numerator * pow(denominator, -1, Q) % Q


simulated = [
    simulate_honest_verifier_transcript(SCHNORR_PUBLIC, random.Random(TEST_SEED + i))
    for i in range(20)
]
assert all(schnorr_verify(SCHNORR_PUBLIC, transcript) for transcript in simulated)

same_nonce_a = schnorr_prove(SCHNORR_SECRET, nonce=8, challenge=2)
same_nonce_b = schnorr_prove(SCHNORR_SECRET, nonce=8, challenge=9)
EXTRACTED_SECRET = extract_from_reused_commitment(same_nonce_a, same_nonce_b)
assert EXTRACTED_SECRET == SCHNORR_SECRET
print("HVZK simulatoru: 20/20; eyni nonce təhlükəsi: secret çıxarıldı")


### 2.3 SNARK, STARK və Bulletproofs: sxemdən asılı xüsusiyyətlər

Bu adlar bütün instansiyalar üçün eyni zəmanət vermir. Məsələn, Groth16 qısa sübut verir,
lakin circuit-specific trusted setup tələb edir; başqa SNARK-larda setup universal və ya
transparent ola bilər. STARK ailələri əsasən heşlərə söykənən transparent sistemlərdir, amma
sübut ölçüsü və verifier xərci konkret sxem və parametrdən asılıdır. Bulletproofs trusted setup
tələb etməyən range-proof ailəsidir; “logaritmik ölçü” ifadəsi bütün ZKP tapşırıqlarına şamil edilmir.

Hər müqayisədə statement dili, setup, prover/verifier xərci, sübut ölçüsü, assumption və
zero-knowledge variantı ayrıca yazılmalıdır.


### ✍️ Çalışma 1 — ZKP

1. Schnorr yoxlama bərabərliyini addım-addım çıxarın.
2. Düzgün transkriptin bir sahəsini dəyişin və rədd nəticəsini göstərin.
3. Eyni commitment üçün iki challenge-in witness-i niyə açdığını izah edin.
4. Simulator kodunun yalnız honest-verifier zero-knowledge intuisiya verdiyini əsaslandırın.
5. Seçdiyiniz SNARK/STARK sxemi üçün setup, assumption və ölçü mənbəyini göstərin.


## 🧮 Homomorf şifrələmə


### 3.1 Anlayış

Homomorf şifrələmədə `Eval` şifrəli mətnlər üzərində funksiya hesablayır və deşifrlənmiş nəticə
açıq mətnlər üzərində həmin funksiyanın nəticəsinə bərabər olur. Dəstəklənən funksiyalar ailəsi,
noise büdcəsi, encoding və təhlükəsizlik modeli sxemə görə dəyişir. Məxfilik avtomatik olaraq
hesablamanın düzgünlüyünü, giriş siyasətini və ya ciphertext autentifikasiyasını təmin etmir.


### 3.2 Paillier-in additiv homomorfizmi

Paillier üçün `E(m₁)·E(m₂) mod n²` deşifrlənəndə `(m₁+m₂) mod n` alınır; `E(m)^k`
isə `km mod n` verir. Aşağıdakı kiçik açar yalnız alqoritmik invariant üçündür. Deterministik
test RNG-si və 9-bit modul təhlükəsiz deyil. Sxem malleable-dir: ciphertext-in idarəli dəyişməsi
plaintext-i də dəyişə bilər, ona görə ayrıca autentifikasiya/protokol nəzarəti lazımdır.


In [ ]:
PAILLIER_P, PAILLIER_Q = 17, 19
PAILLIER_N = PAILLIER_P * PAILLIER_Q
PAILLIER_N2 = PAILLIER_N * PAILLIER_N
PAILLIER_G = PAILLIER_N + 1
PAILLIER_LAMBDA = lcm(PAILLIER_P - 1, PAILLIER_Q - 1)


def paillier_L(value):
    return (value - 1) // PAILLIER_N


PAILLIER_MU = pow(
    paillier_L(pow(PAILLIER_G, PAILLIER_LAMBDA, PAILLIER_N2)) % PAILLIER_N,
    -1,
    PAILLIER_N,
)


def paillier_encrypt(message, rng=None, r=None):
    if not 0 <= message < PAILLIER_N:
        raise ValueError("message Z_n daxilində olmalıdır")
    if r is None:
        if rng is None:
            raise ValueError("toy test üçün rng və ya r verilməlidir")
        r = rng.randrange(1, PAILLIER_N)
        while gcd(r, PAILLIER_N) != 1:
            r = rng.randrange(1, PAILLIER_N)
    if not 1 <= r < PAILLIER_N or gcd(r, PAILLIER_N) != 1:
        raise ValueError("r Z_n* daxilində olmalıdır")
    return pow(PAILLIER_G, message, PAILLIER_N2) * pow(r, PAILLIER_N, PAILLIER_N2) % PAILLIER_N2


def paillier_decrypt(ciphertext):
    value = pow(ciphertext, PAILLIER_LAMBDA, PAILLIER_N2)
    return paillier_L(value) * PAILLIER_MU % PAILLIER_N


paillier_rng = random.Random(TEST_SEED)
PAILLIER_A, PAILLIER_B = 15, 7
c_a = paillier_encrypt(PAILLIER_A, paillier_rng)
c_b = paillier_encrypt(PAILLIER_B, paillier_rng)
c_sum = c_a * c_b % PAILLIER_N2
c_scaled = pow(c_a, 3, PAILLIER_N2)
assert paillier_decrypt(c_sum) == (PAILLIER_A + PAILLIER_B) % PAILLIER_N
assert paillier_decrypt(c_scaled) == (3 * PAILLIER_A) % PAILLIER_N

# Malleability nümunəsi: g ilə vurmaq plaintext-ə 1 əlavə edir.
c_modified = c_a * PAILLIER_G % PAILLIER_N2
assert paillier_decrypt(c_modified) == (PAILLIER_A + 1) % PAILLIER_N
try:
    paillier_encrypt(1, r=PAILLIER_P)
except ValueError:
    INVALID_R_REJECTED = True
else:
    INVALID_R_REJECTED = False
assert INVALID_R_REJECTED
print("Paillier: toplama, skalyar vurma, malleability və invalid-r testləri keçdi")


### 3.3 FHE və bootstrapping

FHE uyğun circuit-lərin ixtiyari dərinlikdə qiymətləndirilməsini hədəfləyir. Bir çox sxemdə
əməliyyatlar ciphertext noise-unu artırır. **Bootstrapping sadəcə adi “yenidən şifrələmə” deyil:**
sxem öz decryption/refresh circuit-ini şifrəli secret-key materialı üzərində homomorf qiymətləndirərək
daha aşağı noise-li ciphertext əldə edir. Hansı circuit-in, açar materialının və security assumption-un
işləndiyi konkret sxemdən asılıdır. Leveled HE seçilmiş dərinlik üçün bootstrapping-siz də qurula bilər.

Performans üçün “klassik şifrədən N dəfə yavaşdır” kimi universal rəqəm düzgün deyil; hardware,
parametrlər, packing, circuit və kitabxana versiyası ilə ölçülmüş benchmark lazımdır.


### 3.4 Tətbiq sərhədləri

HE serverdən plaintext-i gizlədə bilər, lakin nəticənin düzgünlüyünü, access control-u, metadata
məxfiliyini və model/inference hücumlarına davamlılığı özü təmin etmir. Tətbiq dizaynında threat model,
key custody, circuit leakage, doğrulama, resurs limiti və çıxışın nə açıqladığı ayrıca qiymətləndirilir.


### ✍️ Çalışma 2 — Homomorf şifrələmə

1. `E(15)·E(7)` və `E(15)^3` nəticələrini deşifrləyib invariantı göstərin.
2. `g` ilə ciphertext vurulmasının nə üçün malleability nümunəsi olduğunu izah edin.
3. Bootstrapping ilə client-side decrypt-and-reencrypt arasındakı fərqi yazın.
4. Leveled HE üçün tələb olunan circuit dərinliyinin parametr seçiminə təsirini araşdırın.


## ⛓️ Blokçeyn və kriptoqrafik ledger komponentləri


### 4.1 Dəqiq sərhəd

Blok daxilində heşlər və bloklararası keçid dəyişiklikləri görünən edə bilər. Lakin tək heş-zəncir
decentralization, konsensus, finality, availability və ya iqtisadi Sybil müqaviməti yaratmır. Bunlar
şəbəkə və konsensus dizaynından asılı sistem xüsusiyyətləridir. Permissioned ledger mərkəzləşdirilmiş
idarəetməyə malik ola bilər; açıq ledger isə iştirakçıları adətən pseudonym ilə göstərir, anonimlik
zəmanəti vermir. “Immutable” əvəzinə threat model daxilində **tamper-evident/tamper-resistant** deyirik.


### 4.2 Merkle daxilolma sübutu və heş-zəncir

Merkle proof bir yarpağın konkret root-a daxil olduğunu sibling heşləri ilə yoxlayır. O, tranzaksiyanın
hüquqi düzgünlüyünü və ya blokun konsensusla qəbulunu sübut etmir. Aşağıdakı model domain separation
(`0x00` yarpaq, `0x01` daxili node) istifadə edir və odd node-u təkrarlayır; bu, yalnız seçilmiş toy
formatın qaydasıdır.


In [ ]:
def merkle_leaf(transaction):
    return sha256(b"\x00" + transaction.encode("utf-8")).digest()


def merkle_parent(left, right):
    return sha256(b"\x01" + left + right).digest()


def merkle_root(transactions):
    if not transactions:
        raise ValueError("ən azı bir tranzaksiya tələb olunur")
    level = [merkle_leaf(transaction) for transaction in transactions]
    while len(level) > 1:
        if len(level) % 2:
            level.append(level[-1])
        level = [merkle_parent(level[i], level[i + 1]) for i in range(0, len(level), 2)]
    return level[0]


def merkle_proof(transactions, index):
    if not 0 <= index < len(transactions):
        raise IndexError("yarpaq indeksi yanlışdır")
    proof = []
    level = [merkle_leaf(transaction) for transaction in transactions]
    while len(level) > 1:
        if len(level) % 2:
            level.append(level[-1])
        sibling = index ^ 1
        proof.append(("right" if index % 2 == 0 else "left", level[sibling]))
        level = [merkle_parent(level[i], level[i + 1]) for i in range(0, len(level), 2)]
        index //= 2
    return proof


def verify_merkle_proof(transaction, proof, expected_root):
    current = merkle_leaf(transaction)
    for side, sibling in proof:
        current = merkle_parent(current, sibling) if side == "right" else merkle_parent(sibling, current)
    return current == expected_root


def make_block(height, previous_hash, transactions):
    body = {
        "height": height,
        "previous_hash": previous_hash,
        "transactions": list(transactions),
        "merkle_root": merkle_root(transactions).hex(),
    }
    return {**body, "hash": digest_hex(canonical_bytes(body))}


transactions = ["A->B:3", "B->C:1", "C->D:2", "D->A:1"]
MERKLE_ROOT = merkle_root(transactions)
MERKLE_PROOF = merkle_proof(transactions, 2)
assert verify_merkle_proof(transactions[2], MERKLE_PROOF, MERKLE_ROOT)
assert not verify_merkle_proof("C->D:200", MERKLE_PROOF, MERKLE_ROOT)

genesis = make_block(0, "0" * 64, ["genesis"])
second = make_block(1, genesis["hash"], transactions)
chain = [genesis, second]
assert chain[1]["previous_hash"] == chain[0]["hash"]

tampered_body = {key: value for key, value in second.items() if key != "hash"}
tampered_body["transactions"] = ["A->B:300", *transactions[1:]]
assert digest_hex(canonical_bytes(tampered_body)) != second["hash"]
print("Merkle proof: daxilolma qəbul, dəyişiklik rədd; heş-zəncir tamper-i göstərdi")


### 4.3 Konsensus və finality

Proof of Work, Proof of Stake və BFT protokolları eyni attacker, synchrony və finality modelinə malik
deyil. “51%” də bütün sistemlər üçün universal hədd deyil. Təhlildə resurs növü, fork-choice,
validator set, slashing, network assumptions və probabilistic/deterministic finality göstərilməlidir.
Bu notebook konsensus implementasiya etmir və Merkle nəticəsindən konsensus nəticəsi çıxarmır.


### 4.4 Məxfilik və miqyaslama

Rollup, payment channel və sidechain fərqli trust/data-availability modelləridir. “Layer 2 daha
təhlükəsizdir” və ya “zk-rollup tam məxfilik verir” ümumi qayda deyil: validity proof state transition
düzgünlüyünü sübut edə bilər, amma tranzaksiya məlumatı və metadata açıq qala bilər. Məxfilik ayrıca
protokol və disclosure siyasəti tələb edir.


### ✍️ Çalışma 3 — Ledger

1. Merkle proof-un bir sibling heşini dəyişdirib rədd nəticəsini göstərin.
2. Heş-zəncir, rəqəmsal imza və konsensusun verdiyi zəmanətləri ayrı cədvəldə yazın.
3. Seçdiyiniz ledger üçün iştirakçı identifikasiyasını “anonymous/pseudonymous/identified” kimi əsaslandırın.
4. “Immutable” iddiasını konkret attacker və dəyişiklik proseduru ilə daha dəqiq ifadə edin.


## ⚛️ Kvant açar paylanması (QKD)


### 5.1 BB84 və E91

BB84 (Bennett–Brassard, 1984) prepare-and-measure QKD protokoludur. Alice və Bob bazisləri açıq
kanalda müqayisə edir, uyğun mövqeləri sift edir və seçilmiş bitlərdə xəta səviyyəsini qiymətləndirir.
Dinləmə “avtomatik aşkarlanır” demək əvəzinə deyirik: uyğun modeldə müdaxilə statistik səhv yarada
bilər; tərəflər QBER nümunəsinə və security proof-un həddinə görə abort və ya post-processing qərarı verir.

Klassik kanal autentifikasiya olunmalıdır; əks halda man-in-the-middle mümkündür. Sifting-dən sonra
error correction, parameter estimation və privacy amplification tələb olunur. E91 entanglement və
Bell-korrelyasiyalarına əsaslanan ayrıca protokol ailəsidir; implementasiya zəiflikləri ideal modeldən ayrıdır.


In [ ]:
def simulate_bb84(seed, signal_count=8192, intercept_fraction=0.0):
    if not 0.0 <= intercept_fraction <= 1.0:
        raise ValueError("intercept_fraction [0, 1] daxilində olmalıdır")
    rng = random.Random(seed)
    alice_bits = [rng.randrange(2) for _ in range(signal_count)]
    alice_bases = [rng.randrange(2) for _ in range(signal_count)]
    bob_bases = [rng.randrange(2) for _ in range(signal_count)]
    bob_bits = []

    for bit, alice_basis, bob_basis in zip(alice_bits, alice_bases, bob_bases):
        sent_bit, sent_basis = bit, alice_basis
        if rng.random() < intercept_fraction:
            eve_basis = rng.randrange(2)
            eve_bit = bit if eve_basis == alice_basis else rng.randrange(2)
            sent_bit, sent_basis = eve_bit, eve_basis
        measured = sent_bit if bob_basis == sent_basis else rng.randrange(2)
        bob_bits.append(measured)

    sifted = [i for i in range(signal_count) if alice_bases[i] == bob_bases[i]]
    if len(sifted) < 2:
        raise RuntimeError("QBER nümunəsi üçün kifayət qədər sifted bit yoxdur")
    sample_size = len(sifted) // 3
    sample = set(rng.sample(sifted, sample_size))
    errors = sum(alice_bits[i] != bob_bits[i] for i in sample)
    remaining = [i for i in sifted if i not in sample]
    return {
        "signals": signal_count,
        "sifted": len(sifted),
        "sampled": sample_size,
        "qber": errors / sample_size,
        "remaining_raw_bits": len(remaining),
        "post_processing_implemented": False,
    }


BB84_NO_EVE = simulate_bb84(TEST_SEED, intercept_fraction=0.0)
BB84_FULL_INTERCEPT = simulate_bb84(TEST_SEED, intercept_fraction=1.0)
assert BB84_NO_EVE["qber"] == 0.0
assert 0.20 < BB84_FULL_INTERCEPT["qber"] < 0.30
assert BB84_FULL_INTERCEPT["post_processing_implemented"] is False
print("BB84 no-Eve QBER:", BB84_NO_EVE["qber"])
print("BB84 full intercept-resend QBER:", round(BB84_FULL_INTERCEPT["qber"], 4))


### 5.2 Kvant kompüterləri haqqında düzgün nəticə

Fiziki qubit sayı təkbaşına RSA/ECC üçün risk tarixini müəyyən etmir. Shor hücumunun resursu logical
qubit, səhv korreksiyası, gate fidelity, circuit depth, arxitektura və optimallaşdırmadan asılıdır.
Kiçik ədədi klassik və ya toy quantum circuit ilə faktorlaşdırmaq production RSA açarının sındırılma
vaxtını ölçmür. Buna görə bu praktika sabit şirkət/qubit cədvəli və “RSA filan ildə qırılacaq” proqnozu vermir.

Risk idarəetməsi üçün inventar, məlumatın məxfilik ömrü, standards status-u və migration testləri
tarixlə izlənməlidir; PQC keçidi gələcək quantum kompüterinin dəqiq tarixini gözləməməlidir.


### 5.3 Peyk QKD nümunəsi

Micius 2016-cı ildə buraxılmış quantum-science peykidir; sonrakı təcrübələr peyk–yer quantum
rabitəsi və interkontinental QKD imkanlarını nümayiş etdirib. Bu nəticə öz-özlüyündə qlobal quantum
internet, limitsiz məsafə və ya bütün endpoint-lərin təhlükəsizliyini sübut etmir. Hər sistem üçün
link budget, trusted-node modeli, autentifikasiya, cihaz side-channel-ları və key-management ayrıca audit olunur.


### ✍️ Çalışma 4 — QKD

1. No-Eve və full intercept-resend QBER nəticələrini müqayisə edin.
2. Sifting-dən sonra niyə bitləri birbaşa tətbiq açarı adlandırmadığımızı izah edin.
3. Autentifikasiya olunmayan klassik kanalda MITM ssenarisi qurun (yalnız threat-model təsviri).
4. QBER həddinin niyə universal sabit olmadığını security proof və implementasiya ilə əlaqələndirin.
5. Quantum computer risk qiymətləndirməsində physical və logical qubit-i ayırın.


## 🔒 MPC, DID və CBDC: texnologiya ≠ avtomatik məxfilik


### 6.1 Secure Multi-Party Computation (MPC)

MPC tərəflərə girişlərini tam açmadan funksiya hesablamağa imkan verə bilər. Məxfilik və düzgünlük
semi-honest/malicious attacker, collusion həddi, abort/fairness, autentifikasiya və preprocessing
modelindən asılıdır. Secret sharing MPC-nin mümkün komponentidir, lakin təkbaşına bütün protokol deyil;
multisig də ümumən MPC ilə eyni anlayış deyil.


### 6.2 Decentralized Identifiers (DID)

W3C DID identifikator və DID document üçün məlumat modelidir. DID mütləq blokçeyn tələb etmir və
“mərkəzi orqan yoxdur” və ya “istifadəçi bütün məlumatı özündə saxlayır” zəmanətini avtomatik vermir;
controller, DID method, resolver, registry və governance bunu müəyyən edir. Verifiable Credential ayrıca
modeldir. Selective disclosure və ZKP yalnız seçilmiş credential/proof suite bunu dəstəkləyəndə yaranır.


### 6.3 Central Bank Digital Currency (CBDC)

CBDC mərkəzi bank öhdəliyinin rəqəmsal forması üçün sistem və siyasət kateqoriyasıdır, konkret
kriptoqrafik primitive deyil. Privacy, offline payment, AML/KYC, programmability, availability və
intermediary rolları dizayn qərarlarıdır. Ölkə üzrə istifadəçi sayı və buraxılış tarixi tez dəyişdiyi üçün
mənbə və tarixsiz status cədvəli verilmir; layihə təhlili rəsmi cari sənədlə ayrıca aparılmalıdır.


## 🔮 Gələcək trendləri necə qiymətləndirməli


### 7.1 2026-07-21 tarixli metod

Standart, deployed məhsul, tədqiqat prototipi və proqnoz ayrı statuslardır. Məsələn, PQC üçün final
FIPS sənədi normativ fakt ola bilər; müəyyən təşkilatın migration tarixi isə plan; fault-tolerant quantum
computer tarixi proqnozdur. ZKP, HE, MPC və QKD üçün də “geniş tətbiq olunacaq” iddiası ölçülə bilən
adoption meyarı və tarix tələb edir.

Trend kartında bunları yazın: `as_of`, primary source, status, ölçü vahidi, uncertainty, dependency,
security model və claim-i etibarsız edə biləcək hadisə. Universal performans əmsalı və qaçılmaz tarix verməyin.


### ✍️ Çalışma 5 — Trend auditi

1. Bir standard faktı, bir vendor roadmap-i və bir tədqiqat proqnozunu statusla ayırın.
2. Hər iddiaya `as_of` tarixi və primary source əlavə edin.
3. “RSA nə vaxt təhlükəsizliyini itirəcək?” sualını ölçülə bilən risk suallarına parçalayın.
4. HE və ZKP benchmark-ı üçün hardware, parametr, input və versiya sahələrini müəyyən edin.
5. CBDC məxfilik iddiasını iştirakçı, görünən metadata və disclosure qaydası ilə yazın.


### 7.2 Yekun müzakirə

- Hansı demo təhlükəsizlik xüsusiyyətini yalnız ilustrasiya edir, hansını test edir?
- Hansı nəticə konkret threat model olmadan mənasızdır?
- Hansı komponent üçün key/state management əsas riskdir?
- Mənbənin tarixi dəyişəndə notebook-da hansı claim yenidən yoxlanmalıdır?


### 7.3 Qiymətləndirmə rubrikası (10 bal)

| Meyar | Bal |
|---|---:|
| Bütün kod hüceyrələri ardıcıl və xətasız işləyir | 2 |
| ZKP müsbət/mənfi test və scope ilə izah olunur | 2 |
| Paillier homomorfizmi və malleability ayrılır | 2 |
| Merkle/heş-zəncir konsensus kimi təqdim edilmir | 2 |
| BB84 QBER, autentifikasiya və post-processing sərhədi göstərilir | 2 |


## 🖥️ İnteqrasiya edilmiş özünü-yoxlama

Bu hüceyrə interaktiv menyu əvəzinə əvvəlki nəticələrin hamısını avtomatik yoxlayır. Beləliklə
CI və təmiz notebook icrası istifadəçi girişini gözləmir, uğursuz branch isə notebook-u dayandırır.


In [ ]:
SELF_CHECKS = {
    "environment": ENVIRONMENT["external_dependencies"] == 0,
    "run_contract": RUN_CONTRACT["interactive_prompts"] == 0,
    "schnorr_positive": schnorr_verify(SCHNORR_PUBLIC, SCHNORR_TRANSCRIPT),
    "schnorr_tamper_rejected": not schnorr_verify(SCHNORR_PUBLIC, SCHNORR_TAMPERED),
    "schnorr_hvzk_simulator": all(schnorr_verify(SCHNORR_PUBLIC, item) for item in simulated),
    "schnorr_nonce_reuse_extracted": EXTRACTED_SECRET == SCHNORR_SECRET,
    "paillier_addition": paillier_decrypt(c_sum) == 22,
    "paillier_scalar": paillier_decrypt(c_scaled) == 45,
    "paillier_invalid_r_rejected": INVALID_R_REJECTED,
    "merkle_positive": verify_merkle_proof(transactions[2], MERKLE_PROOF, MERKLE_ROOT),
    "merkle_tamper_rejected": not verify_merkle_proof("C->D:200", MERKLE_PROOF, MERKLE_ROOT),
    "bb84_no_eve": BB84_NO_EVE["qber"] == 0.0,
    "bb84_intercept_evidence": 0.20 < BB84_FULL_INTERCEPT["qber"] < 0.30,
    "bb84_scope": BB84_FULL_INTERCEPT["post_processing_implemented"] is False,
}
failed = [name for name, passed in SELF_CHECKS.items() if not passed]
assert not failed, f"Özünü-yoxlama uğursuz oldu: {failed}"
print(f"Özünü-yoxlama: {sum(SELF_CHECKS.values())}/{len(SELF_CHECKS)} keçdi")


## 🏠 Yekun tapşırıq

Aşağıdakılardan birini threat model və test planı ilə hazırlayın:

1. Schnorr toy-modeli üçün transcript test paketi və nonce-reuse izahı;
2. Paillier üzrə encrypted-sum prototipi: input range, malleability və key custody sərhədləri;
3. Merkle inclusion verifier: format specification, positive/negative vectors və consensus non-goal;
4. BB84 simulator genişlənməsi: müxtəlif intercept nisbətləri üzrə çox-seed QBER intervalı.

Tələb olunur: README, versiyalar, təkrar olunan test, ən azı iki mənfi hal, mənbələr və “production deyil”
sərhədi. Real şəxsi məlumat, həqiqi açar və ya icazəsiz şəbəkə hədəfi istifadə etməyin.


## 🚀 Növbəti addımlar

- Standart və məqaləni primary source-dan oxuyun, blog xülasəsini normativ mənbə saymayın.
- Production kitabxanasında təhlükəsiz parametr/API seçimini rəsmi sənədlə yoxlayın.
- Kod review-də yalnız “işləyir” yox, threat model, negative test və non-goal tələb edin.
- Tez dəyişən status və benchmark-ları `as_of` tarixi ilə saxlayın.


## ✅ Məşğələnin tamamlanması

Notebook-u uğurla icra etmək kurs sertifikatı və ya production kriptoqrafiya səriştəsinin avtomatik
sübutu deyil. Tamamlama meyarı rubrika, müəllim qiymətləndirməsi və təqdim olunan layihənin ayrıca
review nəticəsi ilə müəyyən edilir.


## 📚 Primary və normativ mənbələr

- Goldwasser, Micali, Rackoff, *The Knowledge Complexity of Interactive Proof-Systems*, STOC 1985,
  DOI: https://doi.org/10.1145/22145.22178
- ACM, 2012 A.M. Turing Award — Goldwasser və Micali:
  https://amturing.acm.org/award_winners/micali_9954407.cfm
- Gentry, *Fully Homomorphic Encryption Using Ideal Lattices*, STOC 2009;
  bootstrapping izahı üçün IACR ePrint 2011/277: https://eprint.iacr.org/2011/277
- Paillier, *Public-Key Cryptosystems Based on Composite Degree Residuosity Classes*, EUROCRYPT 1999.
- NISTIR 8202, *Blockchain Technology Overview*: https://doi.org/10.6028/NIST.IR.8202
- Bennett və Brassard, *Quantum Cryptography: Public Key Distribution and Coin Tossing*, 1984.
- ETSI, *Quantum Safe Cryptography and Security*, classical-channel authentication bölməsi:
  https://www.etsi.org/images/files/QuantumSafeWhitepaper.pdf
- W3C, *Decentralized Identifiers (DIDs) v1.0*: https://www.w3.org/TR/did-core/

Mənbələr son dəfə 2026-07-21 tarixində yoxlanılıb. Toy kodlar standart implementasiyası deyil.
